# 最適化

担当：岩間

1日（朝・昼・夜）の最小コストの組み合わせを**全探索**で選ぶ。解なしのときは、予算・準備時間・栄養のどれを緩めると解が出るかを返す。

仕様は `docs/team_rules.md` 第3節。optimize の定義セルと、ダミーデータでの動作確認セルを分ける。


In [1]:
# 数値計算（配列での一括計算）
import numpy as np
# 表データの読み込みと加工
import pandas as pd

# 共通の仮定値（config.py）
import config


## 関数の定義

`st.` は書かない。`config` も使わない（引数と戻り値だけで渡す）。後で `.py` に書き出すのはこの2つのセル。

In [2]:
SLOTS = ("morning", "lunch", "dinner")  # 食事の枠（朝・昼・夜の順）
EPS = 1e-9  # 小数の誤差で境界値を落とさないための微小値


def _parse_exclude(value): #形式の統一用関数
    """exclude_slots の1セル（例: "morning,lunch"）を集合にする。空欄は空集合。"""
    # 空欄は CSV では NaN になるので、空文字と合わせて「除外なし」にする
    if pd.isna(value) or str(value).strip() == "":
        return set()
    # カンマで分けて空白を除き、集合にする（後で in 判定しやすい）
    return {s.strip() for s in str(value).split(",") if s.strip()}


def assign_slots(meals_df, kcal_target, slot_ratio, slot_tol): #朝昼夜判断の関数
    """meals_df に can_morning / can_lunch / can_dinner 列（bool）を足して返す。

    kcal が「kcal_target × slot_ratio[枠]」の ±slot_tol（割合）に入り、
    かつ exclude_slots にその枠が含まれない場合に True。
    入力: meals_df（kcal 列は必須、exclude_slots 列は任意）、kcal_target（kcal/日）
    """
    out = meals_df.copy()  # 元の表は書き換えない

    # 手で除外する枠（行ごとの集合）。列がなければ全行「除外なし」
    if "exclude_slots" in out.columns:
        excluded = out["exclude_slots"].map(_parse_exclude)
    else:
        excluded = pd.Series([set()] * len(out), index=out.index)

    for slot in SLOTS:
        target = kcal_target * slot_ratio[slot]  # この枠の目標kcal
        # kcal が 目標±(slot_tol×目標) の範囲内か
        within = (out["kcal"] - target).abs() <= slot_tol * target + EPS
        # 範囲内 かつ 除外されていない（~ は反転、slot=slot はループ変数の固定）
        out[f"can_{slot}"] = within & ~excluded.map(lambda s, slot=slot: slot in s)
    return out

In [3]:
def _update_best(best, mask, primary, secondary, i, cand_j, cand_k): # 昼×夜の最良の組を更新する関数
    """mask を満たす組のうち (primary, secondary) が最小のものに best を更新する。

    best は (primary, secondary, (i, j, k)) または None。同点は先に見つけた方を残す。
    """
    # 条件を満たす組が1つもなければ、更新せず返す
    if not mask.any():
        return best
    # 条件を満たさない組は inf にして、最小値の対象から外す
    p = np.where(mask, primary, np.inf)
    # 第1キー（コストなど）の最小値
    p_min = p.min()
    # 第1キーが最小の組だけ残し、第2キーで比べる
    s = np.where(mask & (primary <= p_min + EPS), secondary, np.inf)
    # 最小の組の位置（昼の番号, 夜の番号）
    jj, kk = np.unravel_index(np.argmin(s), s.shape)
    s_min = s[jj, kk]
    # これまでより良ければ更新。同点なら更新しない（先に見つけた方を残す）
    if best is None or (p_min, s_min) < (best[0], best[1]):
        return (p_min, s_min, (i, cand_j[jj], cand_k[kk]))
    return best



def optimize(meals_df, budget_yen, max_prep_min, nutrition_target, aux_nutrients=()): # 1日分の最適化関数
    """1日分（朝・昼・夜）の最小コストの組み合わせを全探索で選ぶ。

    入力
      meals_df: meals.csv の列 + can_morning / can_lunch / can_dinner 列（assign_slots の出力）。
                金額は円、時間は分、栄養は kcal / g / mg。
                recipe_id 列（任意）があれば、同じ recipe_id の食事は1日に2回選ばない。
      budget_yen: 1日の予算（円）。合計コストがこれ以下。
      max_prep_min: 1日の準備時間の合計の上限（分）。
      nutrition_target: 制約にする栄養 {列名: (下限, 上限)}。None は制限なし。
        例: {"kcal": (2000, 2400), "protein_g": (65, None)}
      aux_nutrients: 制約に入れず、合計だけ表示する栄養の列名のリスト（補助項目）。
    探索
      朝・昼・夜の候補から1食ずつ、3食とも別の食事になる組み合わせをすべて調べる。
      「別の食事」は recipe_id が違うこと（recipe_id 列がなければ meal_id が違うこと）。
      同じレシピの倍率違い（R101_x1.0 と R101_x1.3 など）は、1日に2回選ばない。
      同コストなら準備時間が短い方、それも同じなら meals_df の並びが早い方を選ぶ。
    戻り値（dict）
      feasible: bool
      selected: 選ばれた3食の DataFrame（slot 列つき）。解なしは None。
      total_cost: 合計コスト（円）。解なしは None。
      totals: 1日の合計（cost_yen, prep_min, 制約項目, 補助項目）。解なしは None。
      slot_counts: 枠ごとの候補数。
      reason: 解ありは None。解なしは次の dict
        slot_empty: 候補が0件の枠のリスト
        budget: 予算だけ外したときの最小コスト（円）。それでも解がなければ None
        prep_time: 準備時間の上限だけ外したときに必要な最小の準備時間（分）。同上
        nutrition: 栄養制約だけ外すと解が出るなら True
        multiple_constraints: どの1条件を外しても解が出ない（複数が同時に厳しい）なら True
    """
    # 行番号を0から振り直す（位置で参照するため）
    df = meals_df.reset_index(drop=True)
    # 制約にする栄養の列名
    keys = list(nutrition_target)
    # 列を NumPy 配列にする（一括計算のため）
    cost = df["cost_yen"].to_numpy(float)
    prep = df["prep_min"].to_numpy(float)
    # 栄養の配列（食事 × 栄養項目）
    nut = df[keys].to_numpy(float) if keys else np.zeros((len(df), 0))
    # 栄養の下限（None は -∞ ＝ 制限なし）
    lo = np.array(
        [-np.inf if nutrition_target[k][0] is None else nutrition_target[k][0] for k in keys]
    )
    # 栄養の上限（None は +∞ ＝ 制限なし）
    hi = np.array(
        [np.inf if nutrition_target[k][1] is None else nutrition_target[k][1] for k in keys]
    )

    # 「同じ食事」の判定キー。recipe_id 列があればそれ、なければ行番号（＝meal_id ごと）
    grp = df["recipe_id"].to_numpy() if "recipe_id" in df.columns else np.arange(len(df))
    # 枠ごとの候補の行番号
    cand = {s: np.flatnonzero(df[f"can_{s}"].to_numpy(bool)) for s in SLOTS}
    # 枠ごとの候補数
    slot_counts = {s: int(len(cand[s])) for s in SLOTS}
    # 候補が0件の枠
    slot_empty = [s for s in SLOTS if slot_counts[s] == 0]
    # 候補0件の枠があれば、探索せず「解なし」で返す
    if slot_empty:
        # 候補0件の枠がある場合の戻り値
        return {
            "feasible": False,
            "selected": None,
            "total_cost": None,
            "totals": None,
            "slot_counts": slot_counts,
            "reason": {
                "slot_empty": slot_empty,
                "budget": None,
                "prep_time": None,
                "nutrition": False,
                "multiple_constraints": False,
            },
        }

    # 昼と夜の候補（朝は下のループで1つずつ回す）
    cj, ck = cand["lunch"], cand["dinner"]
    # best_all=全条件 / no_budget=予算なし / no_prep=時間なし の最良
    best_all = best_no_budget = best_no_prep = None
    # 栄養だけが原因か（予算・時間だけなら解が出るか）
    nutrition_is_cause = False  # 栄養だけ外すと解が出るか（出れば True）

    # 朝のメニューを1つ固定し、昼×夜の全組をまとめて計算する
    for i in cand["morning"]:
        # 合計コストの表（行=昼、列=夜）
        c = cost[i] + cost[cj][:, None] + cost[ck][None, :]
        # 合計準備時間の表
        p = prep[i] + prep[cj][:, None] + prep[ck][None, :]
        # 合計栄養の表（昼 × 夜 × 栄養項目）
        tot = nut[i] + nut[cj][:, None, :] + nut[ck][None, :, :]
        # 3食がすべて別の食事か（recipe_id が違う。倍率違いの同じレシピは同じ扱い）
        gi, gj, gk = grp[i], grp[cj], grp[ck]
        distinct = (gj[:, None] != gi) & (gk[None, :] != gi) & (gj[:, None] != gk[None, :])
        # 予算以下か
        ok_b = c <= budget_yen + EPS
        # 準備時間が上限以下か
        ok_p = p <= max_prep_min + EPS
        # 栄養が全項目で下限〜上限に入るか
        ok_n = ((tot >= lo - EPS) & (tot <= hi + EPS)).all(axis=2)

        # 全条件を満たす組の中で、最小コスト（同点は準備時間が短い方）
        best_all = _update_best(best_all, distinct & ok_b & ok_p & ok_n, c, p, i, cj, ck)
        # 予算だけ外した最小コスト ＝ 必要な予算
        best_no_budget = _update_best(best_no_budget, distinct & ok_p & ok_n, c, p, i, cj, ck)
        # 時間だけ外した最小の準備時間 ＝ 必要な時間
        best_no_prep = _update_best(best_no_prep, distinct & ok_b & ok_n, p, c, i, cj, ck)
        # 予算と時間だけで成り立つ組があれば、栄養が原因
        if not nutrition_is_cause:
            nutrition_is_cause = bool((distinct & ok_b & ok_p).any())

    # 解なし：外した条件ごとの結果から理由を作る
    if best_all is None:
        # あと何円（それでも解がなければ None）
        r_budget = None if best_no_budget is None else float(best_no_budget[0])
        # あと何分（それでも解がなければ None）
        r_prep = None if best_no_prep is None else float(best_no_prep[0])
        # 解なしの戻り値（dict）
        return {
            "feasible": False,                 # 解なし
            "selected": None,                  # 選ばれた3食はなし
            "total_cost": None,                # 合計コストもなし
            "totals": None,                    # 1日の合計もなし
            "slot_counts": slot_counts,        # 枠ごとの候補数（原因の手がかり）
            "reason": {
                "slot_empty": [],                  # 候補0件の枠はなし（ここまで来た時点で全枠に候補あり）
                "budget": r_budget,                # 予算だけ外したときの最小コスト（円）。なければ None
                "prep_time": r_prep,               # 時間だけ外したときの最小の準備時間（分）。なければ None
                "nutrition": nutrition_is_cause,   # 栄養だけが原因なら True
                # 予算・時間・栄養のどれを1つ外しても解が出ない＝複数の条件が同時に厳しい
                "multiple_constraints": r_budget is None
                and r_prep is None
                and not nutrition_is_cause,
            },
        }

    # 解あり：選ばれた3食の行番号（朝・昼・夜）
    i, j, k = best_all[2]
    # 表示する列（基本列 + 制約の栄養）
    cols = ["meal_id", "name", "type", "cost_yen", "prep_min"] + keys
    # 補助の栄養項目を足す（重複は除く）
    cols += [a for a in aux_nutrients if a in df.columns and a not in keys]
    # 表に存在しない列は除く
    cols = [c_ for c_ in cols if c_ in df.columns]
    # 選ばれた3食の表
    selected = df.loc[[i, j, k], cols].copy()
    # 先頭に枠名（morning/lunch/dinner）の列を足す
    selected.insert(0, "slot", list(SLOTS))
    selected = selected.reset_index(drop=True)
    # 数値列を合計して1日の合計にする
    totals = selected.drop(columns=["slot", "meal_id", "name", "type"]).sum().to_dict()
    # 解ありの戻り値（dict）
    return {
        "feasible": True,                          # 解あり
        "selected": selected,                      # 選ばれた3食の表（slot 列つき）
        "total_cost": float(totals["cost_yen"]),   # 1日の合計コスト（円）。float にして numpy 型を避ける
        "totals": totals,                          # 1日の合計（コスト・準備時間・栄養）
        "slot_counts": slot_counts,                # 枠ごとの候補数（朝・昼・夜）
        "reason": None,                            # 解ありなので理由はなし
    }


## 動作確認（ダミーデータ）

配分比・目標kcal・栄養の基準値は仮の値。決まったら置き換える。

In [4]:
# ダミーのメニュー表を読む
meals = pd.read_csv("../data/meals.csv")

# 動作確認用。SLOT_RATIO が config で決まっていれば（None でなければ）それを使う。
# 配分比を config から取る
ratio = config.SLOT_RATIO
# 未決（None）があれば仮の値に差し替える
if any(v is None for v in ratio.values()):
    ratio = {"morning": 0.25, "lunch": 0.35, "dinner": 0.40}  # 仮の値（根拠なし）
KCAL_TARGET = 2200  # 仮の値

# 朝・昼・夜の候補フラグ（can_*）を付ける
meals_slot = assign_slots(meals, KCAL_TARGET, ratio, config.SLOT_TOL)
# 枠ごとの候補を確認する（最後の式の値がセルに表示される）
meals_slot[["meal_id", "name", "kcal", "exclude_slots", "can_morning", "can_lunch", "can_dinner"]]


,meal_id,name,kcal,exclude_slots,can_morning,can_lunch,can_dinner
0,m01,牛丼(外食),700,NaN,True,True,True
1,m02,カレーライス(外食),800,morning,False,True,True
2,m03,ラーメン(外食),650,morning,False,True,True
3,m04,コンビニおにぎり2個,380,NaN,False,False,False
4,m05,定食(焼き魚),750,morning,False,True,True
5,m06,親子丼(自炊),650,NaN,True,True,True
6,m07,納豆卵かけご飯(自炊),500,NaN,True,False,False
7,m08,野菜炒め定食(自炊),600,NaN,True,True,False
8,m09,パスタ(自炊),600,NaN,True,True,False
9,m10,鶏むね肉のソテー定食(自炊),620,NaN,True,True,True


In [5]:
# 制約にする栄養 {列名: (下限, 上限)}。None は制限なし
target = {
    "kcal": (1800, 2400),
    "protein_g": (60, None),
    "fat_g": (None, 75),
    "carb_g": (200, 350),
}  # 動作確認用の仮の値。基準値は食事摂取基準（2025年版）で確認してから決める

# 予算1500円・準備時間90分以内で、最小コストの1日を探す
result = optimize(meals_slot, budget_yen=1500, max_prep_min=90,
                  nutrition_target=target, aux_nutrients=config.AUX_NUTRIENTS)
# 解の有無・合計コスト・枠ごとの候補数
print(result["feasible"], result["total_cost"], result["slot_counts"])
# 選ばれた3食の表
result["selected"]


True 730.0 {'morning': 6, 'lunch': 8, 'dinner': 6}


,slot,meal_id,name,type,cost_yen,prep_min,kcal,protein_g,fat_g,carb_g,fiber_g,salt_g,calcium_mg,iron_mg,vitamin_c_mg
0,morning,m06,親子丼(自炊),自炊,250,30,650,30,15,95,3,3.0,70,3.0,6
1,lunch,m09,パスタ(自炊),自炊,200,25,600,18,14,100,4,2.0,40,1.8,5
2,dinner,m10,鶏むね肉のソテー定食(自炊),自炊,280,35,620,40,12,70,6,2.5,60,1.5,35


In [6]:
# 解なしの例：どの条件を緩めると解が出るかを見る
# 予算・時間だけ厳しくして、解なしの理由を見る
for label, budget, prep in [("予算が厳しい", 600, 90), ("時間が厳しい", 1500, 40), ("両方厳しい", 600, 40)]:
    # 栄養条件はそのまま
    r = optimize(meals_slot, budget, prep, target)
    # reason に「あと何円・何分」が入る
    print(label, r["feasible"], r["reason"])


予算が厳しい False {'slot_empty': [], 'budget': 730.0, 'prep_time': None, 'nutrition': True, 'multiple_constraints': False}
時間が厳しい False {'slot_empty': [], 'budget': None, 'prep_time': 55.0, 'nutrition': False, 'multiple_constraints': False}
両方厳しい False {'slot_empty': [], 'budget': None, 'prep_time': None, 'nutrition': False, 'multiple_constraints': True}


## 検算（愚直な全探索との照合）

In [7]:
# 組み合わせ（直積）を作る標準ライブラリ
import itertools


def naive_min_cost(df, budget, max_prep, target):
    """愚直な三重ループ。optimize の最小コストと一致するかの検算用。"""
    # 行番号を0から振り直す
    df = df.reset_index(drop=True)
    # 枠ごとの候補の行番号（optimize と同じ条件）
    cand = {s: [i for i in range(len(df)) if df.loc[i, f"can_{s}"]] for s in SLOTS}
    # 「同じ食事」の判定キー（optimize と同じ。recipe_id があればそれ、なければ行番号）
    grp = df["recipe_id"].to_numpy() if "recipe_id" in df.columns else list(range(len(df)))
    # 列をリストにしておく（実データは候補が多く、組が数十万通りになるため、表から毎回引くと遅すぎる）
    cost, prep = df["cost_yen"].tolist(), df["prep_min"].tolist()
    nut = {key: df[key].tolist() for key in target}
    # 見つかった最小コスト
    best = None
    # 朝×昼×夜のすべての組を1つずつ調べる（遅いが単純）
    for i, j, k in itertools.product(cand["morning"], cand["lunch"], cand["dinner"]):
        # 同じ食事（同じ recipe_id）が重複していたら除外
        if len({grp[i], grp[j], grp[k]}) < 3:
            continue
        # 3食の合計コスト・準備時間
        total = cost[i] + cost[j] + cost[k]
        # 予算・準備時間を超えたら除外
        if total > budget or prep[i] + prep[j] + prep[k] > max_prep:
            continue
        # 栄養が全項目で範囲内か調べる
        ok = True
        for key, (lo, hi) in target.items():
            t = nut[key][i] + nut[key][j] + nut[key][k]
            ok &= (lo is None or t >= lo) and (hi is None or t <= hi)
        # 条件を満たし、これまでより安ければ更新
        if ok and (best is None or total < best):
            best = total
    return best


# 条件を変えて、optimize と愚直な探索の答えを比べる
for budget, prep in [(1500, 90), (800, 90), (600, 90), (1500, 40)]:
    r = optimize(meals_slot, budget, prep, target)
    # 愚直な探索の答え
    expected = naive_min_cost(meals_slot, budget, prep, target)
    # optimize の答え
    got = r["total_cost"]
    # どちらも解なし、または同じコストなら OK。違えば停止
    assert (got is None and expected is None) or got == expected, (budget, prep, got, expected)
print("愚直な全探索と一致")


愚直な全探索と一致


## 検算（倍率違いの同じレシピを重複して選ばない）

自炊の量を変えた行（倍率違い）を足し、`recipe_id` が同じ行を1日に2回選ばないことを確かめる。
倍率違いは元より安くしてあり、`recipe_id` で判定しなければ重複して選ばれてしまう条件にしてある。

In [ ]:
# 自炊だけを取り出し、安い倍率違いの行（同じレシピ）を作る
base = meals_slot.copy()
base["recipe_id"] = base["meal_id"]  # 元の行：recipe_id は meal_id と同じ
twin = base[base["type"] == "自炊"].copy()
twin["meal_id"] = twin["meal_id"] + "_x0.5"  # 別の行として扱う（recipe_id は元のまま）
twin["cost_yen"] = twin["cost_yen"] * 0.5  # 安くして、重複して選ばれやすくする
both = pd.concat([base, twin], ignore_index=True)

for budget, prep in [(1500, 90), (1500, 200), (800, 90)]:
    r = optimize(both, budget, prep, target)
    # 愚直な探索の答えと一致するか
    expected = naive_min_cost(both, budget, prep, target)
    got = r["total_cost"]
    assert (got is None and expected is None) or got == expected, (budget, prep, got, expected)
    if r["feasible"]:
        # 選ばれた3食のレシピがすべて違うか（meal_id から倍率の接尾辞を除いて比べる）
        ids = [m.split("_x")[0] for m in r["selected"]["meal_id"]]
        assert len(set(ids)) == 3, ids
print("倍率違いの重複なし・愚直な全探索と一致")